# Day 4: Fund Performance Analytics

## Task 1: Compute Daily Returns and Annualised Returns


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

data_dir = Path('../data/processed')
nav_df = pd.read_csv(data_dir / 'clean_nav.csv', parse_dates=['date'])
nav_df = nav_df.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

# Compute daily return
nav_df['daily_return'] = nav_df.groupby('amfi_code')['nav'].pct_change()

# Compute annualised return
def compute_annualised_return(group):
    n = group['daily_return'].notna().sum()
    if n == 0: return np.nan
    compounded = (1 + group['daily_return'].dropna()).prod()
    return (compounded ** (252/n)) - 1

annualised_returns = nav_df.groupby('amfi_code').apply(compute_annualised_return).reset_index()
annualised_returns.columns = ['amfi_code', 'annualised_return']

returns_computed_path = data_dir / 'returns_computed.csv'
nav_df.to_csv(returns_computed_path, index=False)
print(f"Saved daily returns to {returns_computed_path}")


Saved daily returns to ..\data\processed\returns_computed.csv


## Task 2: Calculate CAGR for 1yr, 3yr, and Since Inception periods

*Note: The dataset contains 4.5 years of NAV history. We use Since Inception CAGR instead of 5-year. We calculate CAGR using actual trading days (252/n) to avoid calendar day inaccuracies.*


In [2]:
# Calculate CAGR using trading days formula: (NAV_end / NAV_start) ^ (252/n) - 1

def calculate_cagr_for_periods(df):
    results = []
    for code, group in df.groupby('amfi_code'):
        group = group.sort_values('date')
        if group.empty: continue
            
        end_date = group['date'].iloc[-1]
        nav_end = group['nav'].iloc[-1]
        
        def get_nav_at_date(target_date):
            past_data = group[group['date'] <= target_date]
            if past_data.empty: return np.nan, np.nan
            # Return NAV and number of trading days between target and end_date
            idx_start = past_data.index[-1]
            idx_end = group.index[-1]
            n_days = idx_end - idx_start
            return past_data['nav'].iloc[-1], n_days
            
        # 1 year
        nav_1yr, n_1yr = get_nav_at_date(end_date - pd.DateOffset(years=1))
        cagr_1yr = ((nav_end / nav_1yr) ** (252/n_1yr)) - 1 if pd.notna(nav_1yr) and n_1yr > 0 else np.nan
        
        # 3 year
        nav_3yr, n_3yr = get_nav_at_date(end_date - pd.DateOffset(years=3))
        cagr_3yr = ((nav_end / nav_3yr) ** (252/n_3yr)) - 1 if pd.notna(nav_3yr) and n_3yr > 0 else np.nan
        
        # Since Inception
        nav_inception = group['nav'].iloc[0]
        n_inception = len(group) - 1
        cagr_inception = ((nav_end / nav_inception) ** (252/n_inception)) - 1 if n_inception > 0 else np.nan
        
        results.append({
            'amfi_code': code,
            'cagr_1yr': cagr_1yr,
            'cagr_3yr': cagr_3yr,
            'cagr_inception': cagr_inception
        })
        
    return pd.DataFrame(results)

cagr_df = calculate_cagr_for_periods(nav_df)
cagr_path = data_dir / 'cagr_report.csv'
cagr_df.to_csv(cagr_path, index=False)
print(f"Saved CAGR report to {cagr_path}")


Saved CAGR report to ..\data\processed\cagr_report.csv


## Task 3: Compute Sharpe Ratio


In [3]:
# Sharpe = (Rp - Rf) / Std(Rp)
# Rf = 6.5% = 0.065
# Std(Rp) = daily_std * sqrt(252)

sharpe_results = []
Rf = 0.065

for code, group in nav_df.groupby('amfi_code'):
    rp = annualised_returns[annualised_returns['amfi_code'] == code]['annualised_return'].values[0]
    daily_std = group['daily_return'].std()
    
    if pd.notna(rp) and pd.notna(daily_std) and daily_std != 0:
        annualised_std = daily_std * np.sqrt(252)
        sharpe = (rp - Rf) / annualised_std
    else:
        sharpe = np.nan
        
    sharpe_results.append({'amfi_code': code, 'sharpe_ratio': sharpe})

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_path = data_dir / 'sharpe_values.csv'
sharpe_df.to_csv(sharpe_path, index=False)
print(f"Saved Sharpe ratios to {sharpe_path}")


Saved Sharpe ratios to ..\data\processed\sharpe_values.csv


## Task 4: Compute Sortino Ratio


In [4]:
# Sortino = (Rp - Rf) / Downside_Std

sortino_results = []

for code, group in nav_df.groupby('amfi_code'):
    rp = annualised_returns[annualised_returns['amfi_code'] == code]['annualised_return'].values[0]
    
    # Only negative return days
    negative_returns = group[group['daily_return'] < 0]['daily_return']
    downside_daily_std = negative_returns.std()
    
    if pd.notna(rp) and pd.notna(downside_daily_std) and downside_daily_std != 0:
        annualised_downside_std = downside_daily_std * np.sqrt(252)
        sortino = (rp - Rf) / annualised_downside_std
    else:
        sortino = np.nan
        
    sortino_results.append({'amfi_code': code, 'sortino_ratio': sortino})

sortino_df = pd.DataFrame(sortino_results)
sortino_path = data_dir / 'sortino_values.csv'
sortino_df.to_csv(sortino_path, index=False)
print(f"Saved Sortino ratios to {sortino_path}")


Saved Sortino ratios to ..\data\processed\sortino_values.csv


## Task 5: Compute Alpha & Beta vs Benchmark (Nifty 100)


In [5]:
# Load benchmark indices
benchmarks = pd.read_csv(data_dir / 'clean_benchmark_indices.csv', parse_dates=['date'])

# We need Nifty 100 closing values
nifty100 = benchmarks[benchmarks['index_name'] == 'NIFTY100'].copy()
nifty100 = nifty100.sort_values('date').reset_index(drop=True)
nifty100['nifty_return'] = nifty100['close_value'].pct_change()

# Join with nav data
merged_df = pd.merge(nav_df[['amfi_code', 'date', 'daily_return']], 
                     nifty100[['date', 'nifty_return']], 
                     on='date', how='inner')

alpha_beta_results = []

for code, group in merged_df.groupby('amfi_code'):
    clean_group = group.dropna(subset=['daily_return', 'nifty_return'])
    if len(clean_group) > 2:
        slope, intercept, r_value, p_value, std_err = stats.linregress(clean_group['nifty_return'], clean_group['daily_return'])
        beta = slope
        alpha = intercept * 252
    else:
        beta = np.nan
        alpha = np.nan
        
    alpha_beta_results.append({
        'amfi_code': code,
        'alpha': alpha,
        'beta': beta
    })

alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_path = data_dir / 'alpha_beta.csv'
alpha_beta_df.to_csv(alpha_beta_path, index=False)
print(f"Saved Alpha & Beta values to {alpha_beta_path}")


Saved Alpha & Beta values to ..\data\processed\alpha_beta.csv


## Task 6: Compute Maximum Drawdown


In [6]:
# max_dd = min(NAV / running_max - 1)

max_dd_results = []

for code, group in nav_df.groupby('amfi_code'):
    group = group.sort_values('date').reset_index(drop=True)
    running_max = group['nav'].cummax()
    drawdown = (group['nav'] / running_max) - 1
    
    max_dd = drawdown.min()
    
    # Highlight worst drawdown period
    trough_idx = drawdown.idxmin()
    if pd.isna(max_dd):
        trough_date = pd.NaT
        peak_date = pd.NaT
    else:
        trough_date = group.loc[trough_idx, 'date']
        # The peak is the maximum NAV before or on the trough date
        past_data = group.loc[:trough_idx]
        peak_idx = past_data['nav'].idxmax()
        peak_date = group.loc[peak_idx, 'date']
        
    max_dd_results.append({
        'amfi_code': code,
        'max_dd': max_dd,
        'worst_dd_peak_date': peak_date.date(),
        'worst_dd_trough_date': trough_date.date()
    })

max_dd_df = pd.DataFrame(max_dd_results)
max_dd_path = data_dir / 'max_drawdown.csv'
max_dd_df.to_csv(max_dd_path, index=False)
print(f"Saved Maximum Drawdown to {max_dd_path}")


Saved Maximum Drawdown to ..\data\processed\max_drawdown.csv


## Task 7: Build Fund Scorecard (composite score 0-100)


In [7]:
# Score = 30%*(3yr return rank) + 25%*(Sharpe rank) + 20%*(Alpha rank) + 15%*(Expense ratio rank, inverse) + 10%*(Max DD rank, inverse)

# Load fund master for expense ratio
fund_master = pd.read_csv(data_dir / 'clean_fund_master.csv')

# Merge all metrics
scorecard = fund_master[['amfi_code', 'expense_ratio_pct']].copy()
scorecard = scorecard.merge(cagr_df[['amfi_code', 'cagr_3yr']], on='amfi_code', how='left')
scorecard = scorecard.merge(sharpe_df[['amfi_code', 'sharpe_ratio']], on='amfi_code', how='left')
scorecard = scorecard.merge(alpha_beta_df[['amfi_code', 'alpha']], on='amfi_code', how='left')
scorecard = scorecard.merge(max_dd_df[['amfi_code', 'max_dd']], on='amfi_code', how='left')

# Calculate percentiles (0 to 100)
# Higher is better
scorecard['rank_3yr'] = scorecard['cagr_3yr'].rank(pct=True) * 100
scorecard['rank_sharpe'] = scorecard['sharpe_ratio'].rank(pct=True) * 100
scorecard['rank_alpha'] = scorecard['alpha'].rank(pct=True) * 100

# Lower is better (Expense ratio)
scorecard['rank_expense'] = scorecard['expense_ratio_pct'].rank(pct=True, ascending=False) * 100

# Less negative is better (Max DD)
scorecard['rank_maxdd'] = scorecard['max_dd'].rank(pct=True, ascending=True) * 100

# Calculate composite score
scorecard['composite_score'] = (
    0.30 * scorecard['rank_3yr'] +
    0.25 * scorecard['rank_sharpe'] +
    0.20 * scorecard['rank_alpha'] +
    0.15 * scorecard['rank_expense'] +
    0.10 * scorecard['rank_maxdd']
)

# Sort by score
scorecard = scorecard.sort_values('composite_score', ascending=False).reset_index(drop=True)

scorecard_path = data_dir / 'fund_scorecard.csv'
scorecard.to_csv(scorecard_path, index=False)
print(f"Saved Fund Scorecard to {scorecard_path}")
print(scorecard[['amfi_code', 'composite_score']].head())


Saved Fund Scorecard to ..\data\processed\fund_scorecard.csv


   amfi_code  composite_score
0     148567           86.250
1     120505           82.875
2     120843           82.000
3     100033           80.750
4     120504           80.000


## Task 8: Benchmark Comparison Chart & Tracking Error
*Note: We must normalize the NAV and Index values to start at 100 for a true comparison. If we didn't normalize, the Nifty indices (values over 20,000) would completely squash the mutual fund NAVs into indistinguishable flat lines at the bottom of the chart!*


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

data_dir = Path('../data/processed')

# 1. Identify Top 5 Funds
scorecard = pd.read_csv(data_dir / 'fund_scorecard.csv')
top_5_funds = scorecard.head(5)['amfi_code'].tolist()
fund_master = pd.read_csv(data_dir / 'clean_fund_master.csv')
top_5_names = fund_master[fund_master['amfi_code'].isin(top_5_funds)].set_index('amfi_code')['scheme_name'].to_dict()

# 2. Get 3 Years of Data (End date is max date in our dataset)
nav_df = pd.read_csv(data_dir / 'returns_computed.csv', parse_dates=['date'])
end_date = nav_df['date'].max()
start_date = end_date - pd.DateOffset(years=3)

# Filter NAV for top 5 funds
nav_top5 = nav_df[(nav_df['amfi_code'].isin(top_5_funds)) & (nav_df['date'] >= start_date)].copy()

# Filter Benchmarks
benchmarks = pd.read_csv(data_dir / 'clean_benchmark_indices.csv', parse_dates=['date'])
nifty50 = benchmarks[(benchmarks['index_name'] == 'NIFTY50') & (benchmarks['date'] >= start_date)].copy()
nifty100 = benchmarks[(benchmarks['index_name'] == 'NIFTY100') & (benchmarks['date'] >= start_date)].copy()

plt.figure(figsize=(14, 8))

# Plot Nifty 50
if not nifty50.empty:
    nifty50 = nifty50.sort_values('date').reset_index(drop=True)
    nifty50['normalized'] = (nifty50['close_value'] / nifty50['close_value'].iloc[0]) * 100
    plt.plot(nifty50['date'], nifty50['normalized'], label='NIFTY 50', linewidth=3, color='black')

# Plot Nifty 100
if not nifty100.empty:
    nifty100 = nifty100.sort_values('date').reset_index(drop=True)
    nifty100['normalized'] = (nifty100['close_value'] / nifty100['close_value'].iloc[0]) * 100
    plt.plot(nifty100['date'], nifty100['normalized'], label='NIFTY 100', linewidth=3, color='grey', linestyle='--')

# Plot Top 5 Funds
for code in top_5_funds:
    fund_data = nav_top5[nav_top5['amfi_code'] == code].sort_values('date').reset_index(drop=True)
    if not fund_data.empty:
        fund_data['normalized'] = (fund_data['nav'] / fund_data['nav'].iloc[0]) * 100
        plt.plot(fund_data['date'], fund_data['normalized'], label=top_5_names[code])

plt.title('Top 5 Funds vs Benchmarks (3-Year Normalized Performance)')
plt.xlabel('Date')
plt.ylabel('Normalized Value (Base = 100)')
plt.legend()
plt.grid(True, alpha=0.3)

reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)
chart_path = reports_dir / 'benchmark_chart.png'
plt.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved benchmark chart to {chart_path}")

# 3. Compute Tracking Error against Nifty 100
print("\n--- Tracking Error (vs NIFTY 100) ---")
# Nifty 100 daily returns over last 3 years
nifty100['daily_return'] = nifty100['close_value'].pct_change()

for code in top_5_funds:
    fund_data = nav_top5[nav_top5['amfi_code'] == code].sort_values('date')
    
    # Merge on date to align returns properly
    merged = pd.merge(fund_data[['date', 'daily_return']], nifty100[['date', 'daily_return']], on='date', suffixes=('_fund', '_bench')).dropna()
    
    # Tracking Error = std of difference * sqrt(252)
    diff = merged['daily_return_fund'] - merged['daily_return_bench']
    tracking_error = diff.std() * np.sqrt(252)
    
    print(f"{top_5_names[code]}: {tracking_error:.4f} (Annualised)")


Saved benchmark chart to ..\reports\benchmark_chart.png

--- Tracking Error (vs NIFTY 100) ---
Mirae Asset Large Cap Fund - Regular - Growth: 0.1880 (Annualised)
ICICI Pru Midcap Fund - Regular - Growth: 0.2327 (Annualised)
Kotak Flexicap Fund - Regular - Growth: 0.2065 (Annualised)
HDFC Mid-Cap Opportunities Fund - Regular - Growth: 0.2250 (Annualised)
ICICI Pru Bluechip Fund - Direct - Growth: 0.1873 (Annualised)
